# Bern Stipendium — Test Case Playground

Interactively evaluate the Bernese scholarship rules (ABG / ABV) implemented in [`src/bern_stipendium`](../src/bern_stipendium/).

Use this notebook to:
- run the YAML tests programmatically,
- modify a single case inline and recompute,
- batch-evaluate variations (parameter sweeps).

All test cases use the schema documented in [`tests/stipendium_anspruch.yaml`](../src/bern_stipendium/tests/stipendium_anspruch.yaml).

## 1. Bootstrap the tax-benefit system

In [ ]:
import sys, os, copy, logging
from pathlib import Path

# Make `bern_stipendium` importable from the notebook.
REPO_ROOT = Path.cwd().parent
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# openfisca-core calls `log.warning(msg, exc)` with no %-placeholder when our
# package isn't pip-installed. That raises a TypeError inside logging's own
# formatter, which the logging module then dumps to stderr — Jupyter renders
# that as a red error block. Disable logging's internal-error printer so the
# notebook output stays clean. The underlying warning is harmless: package
# metadata simply falls back to defaults.
logging.raiseExceptions = False

import yaml
import pandas as pd
from openfisca_core.simulation_builder import SimulationBuilder

from bern_stipendium import CountryTaxBenefitSystem

tbs = CountryTaxBenefitSystem()
print("Loaded variables:", len(tbs.variables))


## 2. Helpers

`run_case(case)` takes a single YAML test case (a dict with `period`, `input`, optional `output`) and returns a flat dict of computed outputs for inspection.

`check_case(case)` additionally compares against the `output:` block and reports pass/fail per expected variable.

In [ ]:
def _wrap_for_period(case):
    """OpenFisca's programmatic SimulationBuilder requires every input value to
    be wrapped in a {period: value} dict. The YAML test runner does this
    implicitly using the case-level `period` key; we replicate that here so the
    notebook accepts the same flat YAML schema.
    """
    period = str(case["period"])
    wrapped = {}
    for entity_plural, entities in case["input"].items():
        wrapped[entity_plural] = {}
        for eid, fields in entities.items():
            ent = {}
            for k, v in fields.items():
                # Role assignments on group entities are lists of person ids — pass through.
                if entity_plural != "persons" and isinstance(v, list):
                    ent[k] = v
                else:
                    ent[k] = {period: v}
            wrapped[entity_plural][eid] = ent
    return wrapped


def build_simulation(case):
    sb = SimulationBuilder()
    return sb.build_from_dict(tbs, _wrap_for_period(case))


def _all_persons(case):
    return list(case["input"]["persons"].keys())


def run_case(case, variables=None):
    """Compute selected variables for every person in the case.

    If `variables` is None, computes the set declared in `output.persons.<id>`,
    or a sensible default eligibility bundle if no output is declared.
    Returns a DataFrame indexed by person id.
    """
    sim = build_simulation(case)
    period = str(case["period"])
    persons = _all_persons(case)

    if variables is None:
        if "output" in case and "persons" in case["output"]:
            variables = sorted({
                v
                for p in case["output"]["persons"].values()
                for v in p.keys()
            })
        else:
            variables = [
                "hat_stipendienrechtlichen_wohnsitz_bern",
                "erfuellt_persoenlichen_status",
                "ausbildungstyp_anerkannt",
                "ausbildungsstaette_qualifiziert",
                "stipendium_grundsaetzlich_ausgeschlossen",
                "verzicht_auf_anrechnung_eltern",
                "beduerftig",
                "maximale_beitragsdauer_eingehalten",
                "altersgrenze_eingehalten",
                "fehlbetrag",
                "stipendium_quote",
                "stipendium_anspruch",
                "stipendium_betrag",
            ]

    rows = {}
    for var in variables:
        values = sim.calculate(var, period)
        for idx, pid in enumerate(persons):
            rows.setdefault(pid, {})[var] = values[idx]
    return pd.DataFrame.from_dict(rows, orient="index")


def check_case(case):
    """Run the case and verify against the `output` block. Returns (df, results)."""
    df = run_case(case)
    expected = case.get("output", {}).get("persons", {})
    rows = []
    for pid, exp in expected.items():
        for var, want in exp.items():
            got = df.at[pid, var]
            if isinstance(want, bool):
                ok = bool(got) == bool(want)
            elif isinstance(want, (int, float)):
                ok = abs(float(got) - float(want)) < 0.6
            else:
                ok = str(got) == str(want)
            rows.append({"person": pid, "variable": var, "expected": want, "got": got, "pass": ok})
    return df, pd.DataFrame(rows)


## 3. Run all cases from the YAML test file

In [ ]:
TEST_FILE = SRC / "bern_stipendium" / "tests" / "stipendium_anspruch.yaml"
with open(TEST_FILE) as f:
    cases = yaml.safe_load(f)

summary = []
for case in cases:
    _, results = check_case(case)
    summary.append({
        "case": case["name"],
        "checks": len(results),
        "failed": int((~results["pass"]).sum()),
    })
pd.DataFrame(summary)

## 4. Inspect a single case in detail

Pick any case by index and look at the full output.

In [ ]:
case = cases[0]
print("Case:", case["name"])
df, results = check_case(case)
display(df.T)
display(results)

## 5. Build your own case inline

Edit any field below and re-run. The cell is fully self-contained — no YAML needed.

In [ ]:
custom_case = {
    "name": "Custom: Tertiärstudent Wohnsitz Bern, knapp bedürftig",
    "period": 2024,
    "input": {
        "persons": {
            "alex": {
                "wohnsitz_grundlage": "elterlicher_wohnsitz",
                "staatsangehoerigkeit": "schweizer",
                "ausbildungstyp": "erstausbildung",
                "ausbildungsstufe": "tertiaerstufe",
                "ausbildungsstaette_anerkannt": True,
                "aktuelles_ausbildungsjahr": 2,
                "kumulierte_ausbildungsjahre": 1,
                "anerkannte_ausbildungskosten": 26000,
                "eigene_anrechenbare_mittel": 4000,
                "alter": 21,
            }
        },
        "households": {
            "h": {
                "applicants": ["alex"],
                "eltern_anrechenbare_mittel": 8000,
            }
        },
    },
}

run_case(custom_case).T

## 6. Parameter sweep

Vary one input across a range and chart the resulting `stipendium_betrag`. Useful to see thresholds and quote breakpoints.

In [ ]:
def sweep(base_case, person_id, field, values, outputs=("stipendium_anspruch", "fehlbetrag", "stipendium_quote", "stipendium_betrag")):
    rows = []
    for v in values:
        c = copy.deepcopy(base_case)
        c["input"]["persons"][person_id][field] = v
        df = run_case(c, variables=list(outputs))
        row = {field: v, **{o: df.at[person_id, o] for o in outputs}}
        rows.append(row)
    return pd.DataFrame(rows)


sweep_df = sweep(custom_case, "alex", "eigene_anrechenbare_mittel", list(range(0, 30001, 2000)))
sweep_df

In [ ]:
ax = sweep_df.plot(
    x="eigene_anrechenbare_mittel",
    y="stipendium_betrag",
    marker="o",
    title="Stipendium amount vs. own means",
)
ax.set_ylabel("CHF")
ax.grid(True)

## 7. Two-dimensional sweep

Cross-vary two inputs and produce a heatmap of the granted amount.

In [ ]:
def sweep_2d(base_case, person_id, field_a, values_a, field_b, values_b, output="stipendium_betrag", household_id=None, household_field=None):
    grid = pd.DataFrame(index=values_a, columns=values_b, dtype=float)
    for a in values_a:
        for b in values_b:
            c = copy.deepcopy(base_case)
            c["input"]["persons"][person_id][field_a] = a
            if household_id and household_field:
                c["input"]["households"][household_id][household_field] = b
            else:
                c["input"]["persons"][person_id][field_b] = b
            df = run_case(c, variables=[output])
            grid.at[a, b] = df.at[person_id, output]
    grid.index.name = field_a
    grid.columns.name = household_field or field_b
    return grid


grid = sweep_2d(
    custom_case,
    person_id="alex",
    field_a="eigene_anrechenbare_mittel",
    values_a=list(range(0, 20001, 4000)),
    field_b="eltern_anrechenbare_mittel",
    values_b=list(range(0, 20001, 4000)),
    household_id="h",
    household_field="eltern_anrechenbare_mittel",
)
grid

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(grid.values, origin="lower", aspect="auto")
ax.set_xticks(range(len(grid.columns)), grid.columns)
ax.set_yticks(range(len(grid.index)), grid.index)
ax.set_xlabel(grid.columns.name)
ax.set_ylabel(grid.index.name)
ax.set_title("Stipendium amount")
fig.colorbar(im, ax=ax, label="CHF")